# Resolver report

Data comes from `reports/resolver.duckdb`, refreshed by `scripts/report.sh` before this notebook opens.

Run all cells (**Kernel → Restart & Run All**) to see the current state of your benchmark runs.

In [ ]:
from pathlib import Path
import os
import duckdb


def _repo_root():
    here = Path(os.getcwd()).resolve()
    for p in [here, *here.parents]:
        if (p / "go.mod").exists():
            return p
    raise SystemExit("repo root (go.mod) not found — run via scripts/report.sh")


ROOT = _repo_root()
DB = ROOT / "reports" / "resolver.duckdb"
if not DB.exists():
    raise SystemExit(f"{DB} missing — run scripts/report.sh first")

con = duckdb.connect(str(DB), read_only=True)
print(f"connected: {DB}")

## What's in the database?

In [ ]:
con.sql("SHOW TABLES").to_df()

## Runs — one row per benchmark run

`overall` is PASS / PARTIAL / FAIL based on tier gating. `p95_ms` is the 95th-percentile per-query latency.

In [ ]:
con.sql("""
    SELECT run_id, model, resolved_real_model, overall,
           correct_count, query_count, p95_ms
    FROM run_summary
    ORDER BY run_id DESC
""").to_df()

### Model × scenario outcome matrix

Pivoted view of the `comparison` data — rows are models, columns are scenarios, values are the most recent score. Makes it easy to eyeball which models fail which scenarios without scrolling the flat list below.

In [ ]:
# Pivot: one row per model (falling back to virtual `model` when
# `resolved_real_model` is empty, as it is for v1 manifests), one column
# per scenario_id, most recent score per (model, scenario).
df = con.sql("""
    SELECT COALESCE(NULLIF(resolved_real_model, ''), model) AS model,
           scenario_id, score, run_id
    FROM comparison
    ORDER BY run_id DESC
""").to_df()

recent = df.drop_duplicates(subset=["model", "scenario_id"], keep="first")
recent.pivot(index="model", columns="scenario_id", values="score")

## Per-query results

Flat view: one row per (run, scenario). Limited to the most recent 50 rows; edit the SQL to widen the window.

In [ ]:
con.sql("""
    SELECT run_id, tier, model, scenario_id, score, elapsed_ms
    FROM comparison
    ORDER BY run_id DESC, tier, scenario_id
    LIMIT 50
""").to_df()

## Community benchmarks

Reference scores from the broader LLM ecosystem. `model_key` is the normalised form used to join against `run_summary.resolved_real_model`.

In [ ]:
con.sql("""
    SELECT model, model_key, benchmark, metric, value, as_of
    FROM community_benchmarks
    ORDER BY model, benchmark
""").to_df()

### Model × benchmark matrix

Pivoted view — rows are normalised `model_key`, columns are `benchmark/metric` pairs. One value per cell.

In [ ]:
# Pivot: one row per model_key (normalised name so quant / casing /
# org-prefix variants collapse), one column per benchmark/metric pair,
# values are the published reference scores.
df = con.sql("""
    SELECT model_key,
           benchmark || '/' || metric AS bench_metric,
           value
    FROM community_benchmarks
""").to_df()

df.pivot_table(index="model_key", columns="bench_metric", values="value", aggfunc="first")

## Next

- Explore a single run in **`reproducibility.ipynb`**.
- Write your own SQL below — `con` is a read-only DuckDB connection and `.to_df()` renders a pandas DataFrame.
- Re-run `scripts/report.sh` after new benchmark runs to refresh.